# Trace Generation — Qwen3-8B on P100

Pipeline test: generate TIR traces with Qwen3-8B on P100 (16GB).
- 16 samples per problem with logprobs
- Standard chat API (no Harmony format)
- `/no_think` mode for speed
- Resume-safe: each problem saved as individual JSON

In [ ]:
import warnings; warnings.simplefilter('ignore')
import os, sys, subprocess, json, re, math, time, queue, threading, contextlib, glob
from pathlib import Path
from collections import Counter

In [ ]:
# Install vLLM — adjust this cell based on what's available
# Option A: from aimo-3-utils wheels (if kernel source is set)
# Option B: pip install (if internet is enabled)
try:
    import vllm
    print(f'vLLM already installed: {vllm.__version__}')
except ImportError:
    # Try aimo-3-utils wheels first
    wheels_archive = '/kaggle/input/aimo-3-utils/wheels.tar.gz'
    if os.path.exists(wheels_archive):
        tmp = '/kaggle/tmp/setup'
        os.makedirs(tmp, exist_ok=True)
        subprocess.run(['tar', '-xzf', wheels_archive, '-C', tmp], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index',
                        '--find-links', f'{tmp}/wheels', 'vllm'], check=True)
    else:
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'vllm'], check=True)
    import vllm
    print(f'vLLM installed: {vllm.__version__}')

In [ ]:
for k, v in [('TRANSFORMERS_NO_TF', '1'), ('TRANSFORMERS_NO_FLAX', '1'), ('CUDA_VISIBLE_DEVICES', '0'),
             ('TOKENIZERS_PARALLELISM', 'false')]:
    os.environ[k] = v

In [ ]:
from jupyter_client import KernelManager
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
from openai import OpenAI

## Configuration

In [ ]:
class CFG:
    # === MODEL — CHANGE THIS ===
    # Point to your uploaded Qwen3-8B model on Kaggle
    # e.g. '/kaggle/input/qwen3-8b/transformers/default/1'
    # or '/kaggle/input/your-dataset-name/Qwen3-8B'
    model_path = '/kaggle/input/qwen3-8b'  # <-- UPDATE THIS
    served_model_name = 'qwen3-8b'

    # Trace generation
    n_samples = 16
    max_turns = 12
    temperature = 0.7
    top_logprobs = 10        # top-10 for better entropy estimates (research: 5 min, 20 ideal)

    # Prompts
    system_prompt = ('You are a world-class math olympiad solver. '
                     'Solve the problem step by step. Use Python code when helpful — '
                     'wrap code in ```python blocks and I will execute it for you. '
                     'The final answer must be a non-negative integer between 0 and 99999. '
                     'Place your final integer answer inside \\boxed{}.')
    preference_prompt = ('You have access to Python with math, numpy, and sympy. '
                         'Use code to verify your reasoning. /no_think')

    # Timing
    server_timeout = 300   # P100 loads slower
    sample_timeout = 300
    jupyter_timeout = 10
    time_limit = 8.5 * 3600  # 8.5 hours

    # vLLM — P100 settings (16GB VRAM)
    context_tokens = 8192    # shorter context to save memory
    buffer_tokens = 256
    gpu_memory_utilization = 0.92
    batch_size = 32          # smaller batch for P100
    dtype = 'float16'        # P100 doesn't support bfloat16
    min_p = 0.02
    seed = 42

    # Dataset
    problems_csv = '/kaggle/input/aimo3-curated-problems/problems.csv'

    # Output
    output_dir = '/kaggle/working/traces'

os.makedirs(CFG.output_dir, exist_ok=True)
print(f'Model: {CFG.model_path}')
print(f'Generating {CFG.n_samples} samples/problem, max {CFG.max_turns} turns')
print(f'Context: {CFG.context_tokens} tokens, dtype: {CFG.dtype}')
print(f'Top logprobs: {CFG.top_logprobs}')
print(f'Time limit: {CFG.time_limit/3600:.1f}h')

## Load Problems

In [ ]:
df = pd.read_csv(CFG.problems_csv)
PROBLEMS = []
for _, row in df.iterrows():
    answer = row.get('answer', '')
    try:
        answer = int(float(answer))
    except (ValueError, TypeError):
        answer = None
    PROBLEMS.append({
        'id': str(row['id']),
        'problem': str(row['problem']),
        'answer': answer,
        'topic': str(row.get('topic', '')),
        'source': str(row.get('source', '')),
    })

# Resume support: skip completed problems
completed = set()
for f in Path(CFG.output_dir).glob('problem_*.json'):
    try:
        with open(f) as fh:
            data = json.load(fh)
            if len(data.get('samples', [])) == CFG.n_samples:
                completed.add(data['problem_id'])
    except:
        pass

remaining = [p for p in PROBLEMS if p['id'] not in completed]
print(f'Total: {len(PROBLEMS)} | Completed: {len(completed)} | Remaining: {len(remaining)}')

## Sandbox

In [ ]:
class Sandbox:
    _port_lock, _next_port = threading.Lock(), 50000

    @classmethod
    def _get_next_ports(cls, count=5):
        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count
            return ports

    def __init__(self, timeout=10):
        self._timeout = timeout
        ports = self._get_next_ports(5)
        env = os.environ.copy()
        env.update({'PYDEVD_DISABLE_FILE_VALIDATION': '1', 'PYTHONWARNINGS': 'ignore', 'MPLBACKEND': 'Agg'})
        self._km = KernelManager()
        self._km.shell_port, self._km.iopub_port, self._km.stdin_port, self._km.hb_port, self._km.control_port = ports
        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])
        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._timeout)
        self.execute('import math, numpy, sympy, itertools, collections')

    def execute(self, code, timeout=None):
        t = timeout or self._timeout
        msg_id = self._client.execute(code, store_history=True, allow_stdin=False, stop_on_error=False)
        stdout, stderr, start = [], [], time.time()
        while True:
            if time.time() - start > t:
                self._km.interrupt_kernel()
                return f'[ERROR] Timed out after {t}s'
            try:
                msg = self._client.get_iopub_msg(timeout=1.0)
            except queue.Empty:
                continue
            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue
            mt, c = msg.get('msg_type'), msg.get('content', {})
            if mt == 'stream':
                (stdout if c.get('name') == 'stdout' else stderr).append(c.get('text', ''))
            elif mt == 'error':
                tb = c.get('traceback', [])
                stderr.append(''.join(re.sub(r'\x1b\[[0-9;]*m', '', f) for f in tb))
            elif mt in ('execute_result', 'display_data'):
                if txt := c.get('data', {}).get('text/plain'):
                    stdout.append(txt if txt.endswith('\n') else f'{txt}\n')
            elif mt == 'status' and c.get('execution_state') == 'idle':
                break
        out, err = ''.join(stdout), ''.join(stderr)
        return f'{out.rstrip()}\n{err}' if err and out else (err or out or '[No output]')

    def reset(self):
        self.execute('%reset -f\nimport math, numpy, sympy, itertools, collections')

    def close(self):
        with contextlib.suppress(Exception):
            self._client.stop_channels()
        with contextlib.suppress(Exception):
            self._km.shutdown_kernel(now=True)

## Trace Generator

In [ ]:
class TraceGenerator:
    def __init__(self, cfg, port=8000):
        self.cfg = cfg
        self.port = port
        self._start_server()
        self.client = OpenAI(base_url=f'http://0.0.0.0:{port}/v1', api_key='none', timeout=600)
        self._wait_for_server()
        self.sandbox = Sandbox(timeout=cfg.jupyter_timeout)
        print('Ready')

    def _start_server(self):
        cmd = [sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
               '--seed', str(self.cfg.seed),
               '--model', self.cfg.model_path,
               '--served-model-name', self.cfg.served_model_name,
               '--tensor-parallel-size', '1',
               '--max-num-seqs', str(self.cfg.batch_size),
               '--gpu-memory-utilization', str(self.cfg.gpu_memory_utilization),
               '--host', '0.0.0.0', '--port', str(self.port),
               '--dtype', self.cfg.dtype,
               '--max-model-len', str(self.cfg.context_tokens),
               '--disable-log-stats', '--enable-prefix-caching']
        print(f'Starting vLLM: {" ".join(cmd)}')
        self.log_file = open('vllm_server.log', 'w')
        self.server = subprocess.Popen(cmd, stdout=self.log_file, stderr=subprocess.STDOUT, start_new_session=True)

    def _wait_for_server(self):
        print('Waiting for vLLM server...')
        start = time.time()
        for _ in range(self.cfg.server_timeout):
            if self.server.poll() is not None:
                log = open('vllm_server.log').read()[-2000:]
                raise RuntimeError(f'Server died:\n{log}')
            try:
                self.client.models.list()
                print(f'Server ready in {time.time()-start:.1f}s')
                return
            except:
                time.sleep(1)
        raise RuntimeError('Server timeout')

    def _extract_code_blocks(self, text):
        """Extract ```python ... ``` blocks from model output."""
        return re.findall(r'```python\s*\n(.*?)```', text, re.DOTALL)

    def _scan_for_answer(self, text):
        for pattern in [r'\\boxed\s*\{\s*([0-9,]+)\s*\}', r'final\s+answer\s+is\s*([0-9,]+)']:
            if matches := re.findall(pattern, text, re.IGNORECASE):
                try:
                    val = int(matches[-1].replace(',', ''))
                    if 0 <= val <= 99999: return val
                except: pass
        return None

    def _compute_entropy(self, logprobs_list):
        """Compute mean Shannon entropy from a list of top-K logprob dicts."""
        if not logprobs_list: return float('inf')
        total, count = 0.0, 0
        for lp in logprobs_list:
            if isinstance(lp, dict) and lp:
                ent = sum(-math.exp(v) * math.log2(max(math.exp(v), 1e-30))
                          for v in lp.values() if math.exp(v) > 0)
                total += ent
                count += 1
        return total / count if count else float('inf')

    def _compute_per_token_entropy(self, logprobs_list):
        """Compute entropy at each token position. Returns list of floats."""
        entropies = []
        for lp in logprobs_list:
            if isinstance(lp, dict) and lp:
                ent = sum(-math.exp(v) * math.log2(max(math.exp(v), 1e-30))
                          for v in lp.values() if math.exp(v) > 0)
                entropies.append(ent)
        return entropies

    def _compute_ngram_rep(self, text, n=4):
        """Compute word-level n-gram repetition ratio. 0=no repetition, 1=all repeated."""
        words = text.split()
        if len(words) < n: return 0.0
        ngrams = [tuple(words[i:i+n]) for i in range(len(words)-n+1)]
        return 1.0 - len(set(ngrams)) / len(ngrams) if ngrams else 0.0

    def generate_sample(self, problem_text, sample_idx):
        """Generate one sample trace with comprehensive metadata.

        Stores per-token signals for: PRIME (implicit PRM), DPO/KTO (preference),
        iw-SFT (importance weighting), Self-Certainty, DeepConf, PRM step rewards.
        """
        start = time.time()
        self.sandbox.reset()

        messages = [
            {'role': 'system', 'content': self.cfg.system_prompt},
            {'role': 'user', 'content': f'{problem_text}\n\n{self.cfg.preference_prompt}'}
        ]

        trace = {'turns': [], 'logprobs': []}
        answer = None
        full_text = ''

        # Accumulate across all turns
        cumulative_logprob = 0.0
        total_completion_tokens = 0
        total_prompt_tokens = 0
        chosen_logprobs_list = []      # per-token chosen logprob (exact from API)
        finish_reason = None

        for turn in range(self.cfg.max_turns):
            try:
                response = self.client.chat.completions.create(
                    model=self.cfg.served_model_name,
                    messages=messages,
                    temperature=self.cfg.temperature,
                    max_tokens=self.cfg.context_tokens // 2,
                    seed=int((self.cfg.seed + sample_idx) ** 2),
                    logprobs=True,
                    top_logprobs=self.cfg.top_logprobs,
                    extra_body={'min_p': self.cfg.min_p},
                )
            except Exception as e:
                trace['error'] = str(e)
                break

            choice = response.choices[0]
            assistant_text = choice.message.content or ''
            full_text += assistant_text
            finish_reason = choice.finish_reason

            # Token counts from usage
            if response.usage:
                total_prompt_tokens = response.usage.prompt_tokens
                total_completion_tokens += response.usage.completion_tokens

            # Collect logprobs — both top-K dicts AND chosen-token logprob
            turn_logprobs = []
            turn_cumlogprob = 0.0
            turn_tokens = 0
            turn_chosen_lps = []
            if choice.logprobs and choice.logprobs.content:
                for token_lp in choice.logprobs.content:
                    turn_tokens += 1
                    # Chosen token's exact logprob (from API, not approximated)
                    turn_cumlogprob += token_lp.logprob
                    turn_chosen_lps.append(token_lp.logprob)
                    if token_lp.top_logprobs:
                        turn_logprobs.append({t.token: t.logprob for t in token_lp.top_logprobs})

            turn_entropy = self._compute_entropy(turn_logprobs)
            trace['turns'].append({
                'role': 'assistant', 'content': assistant_text,
                'tokens': turn_tokens, 'entropy': turn_entropy,
                'cumulative_logprob': turn_cumlogprob,
            })
            trace['logprobs'].extend(turn_logprobs)
            cumulative_logprob += turn_cumlogprob
            chosen_logprobs_list.extend(turn_chosen_lps)

            # Check for answer
            answer = self._scan_for_answer(assistant_text)
            if answer is not None:
                break

            # Check for code blocks to execute
            code_blocks = self._extract_code_blocks(assistant_text)
            if not code_blocks:
                if choice.finish_reason == 'stop':
                    break
                continue

            # Execute code blocks
            outputs = []
            any_error = False
            for code in code_blocks:
                result = self.sandbox.execute(code)
                outputs.append(result)
                if '[ERROR]' in result or '[No output]' in result:
                    any_error = True

            exec_output = '\n'.join(outputs)
            code_success = not any_error
            trace['turns'].append({
                'role': 'tool', 'name': 'python', 'content': exec_output,
                'code_success': code_success,
            })

            # Feed output back to model
            messages.append({'role': 'assistant', 'content': assistant_text})
            messages.append({'role': 'user', 'content': f'Code output:\n```\n{exec_output}\n```\nContinue solving. /no_think'})

        # ============================================================
        # Per-sample metadata — comprehensive for all downstream uses
        # ============================================================

        # --- Core ---
        trace['answer'] = answer
        trace['finish_reason'] = finish_reason
        trace['time'] = time.time() - start
        trace['n_turns'] = len([t for t in trace['turns'] if t['role'] == 'assistant'])

        # --- Logprob signals (PRIME, DPO, KTO, iw-SFT, Self-Certainty) ---
        trace['cumulative_logprob'] = cumulative_logprob
        trace['completion_tokens'] = total_completion_tokens
        trace['prompt_tokens'] = total_prompt_tokens
        trace['length_normalized_logprob'] = (
            cumulative_logprob / total_completion_tokens if total_completion_tokens > 0 else float('-inf'))
        trace['chosen_logprobs'] = chosen_logprobs_list  # per-token exact, for PRIME pi_theta

        # --- Entropy signals (DeepConf, PRM step rewards, Think Just Enough) ---
        per_token_ents = self._compute_per_token_entropy(trace['logprobs'])
        trace['entropy'] = sum(per_token_ents) / len(per_token_ents) if per_token_ents else float('inf')
        trace['per_token_entropy'] = per_token_ents  # full trajectory for DeepConf group analysis
        if per_token_ents:
            trace['entropy_std'] = (sum((e - trace['entropy'])**2 for e in per_token_ents) / len(per_token_ents)) ** 0.5
            trace['entropy_min'] = min(per_token_ents)
            trace['entropy_max'] = max(per_token_ents)
            sorted_ents = sorted(per_token_ents)
            p10_idx = max(0, len(sorted_ents) // 10 - 1)
            trace['entropy_p10'] = sorted_ents[p10_idx]  # DeepConf bottom-10%
        else:
            trace['entropy_std'] = 0.0
            trace['entropy_min'] = float('inf')
            trace['entropy_max'] = 0.0
            trace['entropy_p10'] = float('inf')

        # --- Code execution signals (CodePRM, TIR preference, GRPO format reward) ---
        trace['code_calls'] = sum(1 for t in trace['turns'] if t.get('role') == 'tool')
        trace['code_errors'] = sum(
            1 for t in trace['turns'] if t.get('role') == 'tool' and not t.get('code_success', True))
        trace['code_success_rate'] = (
            (trace['code_calls'] - trace['code_errors']) / trace['code_calls']
            if trace['code_calls'] > 0 else 1.0)

        # --- Answer stability (TIR preference, quality filtering) ---
        all_turn_answers = []
        full_text_parts = []
        for t in trace['turns']:
            if t['role'] == 'assistant':
                full_text_parts.append(t['content'])
                ans = self._scan_for_answer(t['content'])
                if ans is not None:
                    all_turn_answers.append(ans)
        trace['all_turn_answers'] = all_turn_answers
        trace['answer_changed_count'] = sum(
            1 for i in range(1, len(all_turn_answers)) if all_turn_answers[i] != all_turn_answers[i-1])

        # --- Format validity (GRPO format reward, DPO-VP) ---
        full_text_combined = ''.join(full_text_parts)
        trace['answer_format_valid'] = bool(re.search(r'\\boxed\s*\{', full_text_combined))

        # --- Degeneration detection (repetition filtering) ---
        trace['ngram_rep_4'] = self._compute_ngram_rep(full_text_combined, n=4)

        return trace

    def generate_traces(self, problem):
        """Generate all samples for a problem. Saves JSON immediately."""
        problem_id = problem['id']
        safe_id = re.sub(r'[^a-zA-Z0-9_]', '_', problem_id)
        output_path = Path(self.cfg.output_dir) / f'problem_{safe_id}.json'
        ground_truth = problem.get('answer')

        print(f"\n{'='*60}")
        print(f'Problem: {problem_id} | GT: {ground_truth}')
        print(f"{'='*60}")

        samples = []
        for i in range(self.cfg.n_samples):
            trace = self.generate_sample(problem['problem'], i)
            samples.append(trace)
            status = 'OK' if trace['answer'] == ground_truth else ('WRONG' if trace['answer'] else 'NONE')
            print(f"  [{i+1}/{self.cfg.n_samples}] ans={trace['answer']} "
                  f"ent={trace['entropy']:.2f} clp={trace['cumulative_logprob']:.1f} "
                  f"toks={trace['completion_tokens']} turns={trace['n_turns']} "
                  f"code={trace['code_calls']}/{trace['code_errors']}err "
                  f"rep4={trace['ngram_rep_4']:.2f} "
                  f"t={trace['time']:.0f}s [{status}]")

        # --- Per-sample correctness label (needed by ALL training methods) ---
        for s in samples:
            if ground_truth is not None and s['answer'] is not None:
                s['is_correct'] = (s['answer'] == ground_truth)
            else:
                s['is_correct'] = None

        # --- Problem-level aggregates (GRPO, iw-SFT difficulty weighting) ---
        answers = [s['answer'] for s in samples if s['answer'] is not None]
        votes = Counter(answers)
        n_correct = sum(1 for s in samples if s.get('is_correct') is True)

        output = {
            'problem_id': problem_id,
            'problem': problem['problem'],
            'ground_truth': ground_truth,
            'topic': problem.get('topic', ''),
            'source': problem.get('source', ''),
            'samples': samples,
            'model': self.cfg.served_model_name,
            'n_samples': self.cfg.n_samples,
            'temperature': self.cfg.temperature,
            'top_logprobs': self.cfg.top_logprobs,
            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
            # Problem-level aggregates
            'n_correct': n_correct,
            'pass_rate': n_correct / len(samples) if samples else 0.0,
            'answer_distribution': dict(votes),
        }

        # Compact JSON (no indent) — traces can be large with per-token data
        with open(output_path, 'w') as f:
            json.dump(output, f)
        print(f'  Saved: {output_path}')

        if answers:
            top = votes.most_common(1)[0][0]
            correct = top == ground_truth if ground_truth is not None else 'N/A'
            print(f'  Majority={top} Votes={dict(votes)} '
                  f'Correct={correct} PassRate={n_correct}/{len(samples)}')
        return output

    def cleanup(self):
        self.sandbox.close()
        self.server.terminate()
        self.server.wait()
        self.log_file.close()

## Run

In [ ]:
generator = TraceGenerator(CFG)

In [ ]:
notebook_start = time.time()
done = 0

for i, problem in enumerate(remaining):
    elapsed = time.time() - notebook_start
    if elapsed > CFG.time_limit:
        print(f'\n*** TIME LIMIT ({elapsed/3600:.1f}h) ***')
        break

    print(f'\n--- [{i+1}/{len(remaining)}] {problem["id"]} ---')
    try:
        generator.generate_traces(problem)
        done += 1
    except Exception as e:
        print(f'  ERROR: {e}')
        safe_id = re.sub(r'[^a-zA-Z0-9_]', '_', problem['id'])
        with open(f'{CFG.output_dir}/error_{safe_id}.json', 'w') as f:
            json.dump({'problem_id': problem['id'], 'error': str(e)}, f)

    elapsed = time.time() - notebook_start
    rate = (i + 1) / (elapsed / 60)
    eta = (len(remaining) - i - 1) / rate if rate > 0 else 0
    print(f'  Progress: {done} done | {elapsed/60:.0f}min elapsed | ~{eta:.0f}min remaining')

total_files = len(list(Path(CFG.output_dir).glob('problem_*.json')))
print(f'\n{"="*60}')
print(f'Session: {done} problems in {(time.time()-notebook_start)/60:.1f}min')
print(f'Total completed (all sessions): {total_files}/{len(PROBLEMS)}')
print(f'{"="*60}')

In [ ]:
# Summary
trace_files = sorted(glob.glob(f'{CFG.output_dir}/problem_*.json'))
correct, wrong, no_ans = 0, 0, 0
for tf in trace_files:
    with open(tf) as f:
        data = json.load(f)
    gt = data.get('ground_truth')
    answers = [s['answer'] for s in data['samples'] if s['answer'] is not None]
    if not answers or gt is None:
        no_ans += 1
        continue
    majority = Counter(answers).most_common(1)[0][0]
    if majority == gt: correct += 1
    else: wrong += 1

print(f'Traces: {len(trace_files)} | Correct: {correct} | Wrong: {wrong} | No answer/GT: {no_ans}')
if correct + wrong > 0:
    print(f'Accuracy: {correct/(correct+wrong)*100:.1f}%')

In [ ]:
generator.cleanup()
print('Done')